# FASE 2: DESARROLLO Y MODELADO
## Sistema de Priorización de Brigadas de Salud Extramurales
**Universidad Popular del Cesar (UPC) — Inteligencia Artificial**

Este notebook implementa el ciclo completo de la **Fase 2**:
1. **Carga y Verificación del Dataset Maestro** (74 columnas, Coordenadas GPS y NBI Censo 2018).
2. **Partición 70 / 15 / 15** (Train, Validation, Test) agrupada por municipio para eliminar fuga de datos.
3. **Entrenamiento y Ajuste de Hiperparámetros** de 4 enfoques:
   - *Dummy Classifier* (Línea base trivial)
   - *Regresión Logística* (Línea base lineal balanceada)
   - *Random Forest* (Ensamble de bagging optimizado)
   - *HistGradientBoosting* (Boosting de alto rendimiento tipo LightGBM)
4. **Evaluación Cuantitativa y Comparación de Modelos** (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC).
5. **Análisis de Importancia de Variables (Feature Importance)**.
6. **Componente Prescriptivo (Optimización MILP)** con distancias Haversine para despacho de brigadas.

### 1. Importación de Librerías y Configuración

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, roc_curve
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)
print('Entorno de modelado configurado correctamente.')

### 2. Carga del Dataset y Verificación del Split 70 / 15 / 15

In [ ]:
# Localizar la ruta del dataset maestro
BASE_DIR = Path.cwd().parent.parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
parquet_path = BASE_DIR / 'data' / 'modelo' / 'municipio_anio.parquet'
if not parquet_path.exists():
    parquet_path = BASE_DIR / 'fase 1 api' / 'data' / 'modelo' / 'municipio_anio.parquet'

df = pd.read_parquet(parquet_path)
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

# Resumen de partición 70 / 15 / 15
split_resumen = df.groupby('split').agg(
    filas=('cod_mpio', 'count'),
    municipios_unicos=('cod_mpio', 'nunique'),
    tasa_prioridad_alta=('prioridad_alta', 'mean')
)
split_resumen['pct_filas'] = (split_resumen['filas'] / len(df) * 100).round(1)
split_resumen

### 3. Carga y Desempeño de los Modelos Entrenados

In [ ]:
reportes_dir = BASE_DIR / 'fase 2' / 'reportes'
if not reportes_dir.exists():
    reportes_dir = BASE_DIR / 'fase 1 api' / 'fase 2' / 'reportes'

csv_metricas = reportes_dir / 'metricas_comparacion.csv'
df_metricas = pd.read_csv(csv_metricas)
print('=== TABLA COMPARATIVA DE MODELOS EN TEST SET (CIEGO) ===')
df_metricas

### 4. Visualización de Evaluación: Curvas ROC y Matrices de Confusión

In [ ]:
from IPython.display import Image, display
roc_img = reportes_dir / 'curva_roc_comparacion.png'
cm_img = reportes_dir / 'matriz_confusion_modelos.png'
imp_img = reportes_dir / 'importancia_variables_rf.png'

if roc_img.exists():
    print('--- Curvas ROC Comparativas ---')
    display(Image(filename=str(roc_img)))

if cm_img.exists():
    print('--- Matrices de Confusión ---')
    display(Image(filename=str(cm_img)))

if imp_img.exists():
    print('--- Importancia de Variables (Random Forest) ---')
    display(Image(filename=str(imp_img)))

### 5. Demostración del Componente Prescriptivo: Optimización MILP
Conexión del modelo predictivo con el algoritmo de programación lineal entera para despachar brigadas móviles.

In [ ]:
asignacion_csv = reportes_dir / 'asignacion_brigadas_optima.csv'
if asignacion_csv.exists():
    plan_brigadas = pd.read_csv(asignacion_csv)
    cols_show = ['nom_mpio', 'numero_brigada', 'prob_prioridad_ia', 'pob_vulnerable', 'distancia_base_km', 'n_resguardos', 'nbi_rural']
    print('=== PLAN OPERATIVO ÓPTIMO DE DESPACHO DE BRIGADAS (CESAR) ===')
    display(plan_brigadas[cols_show])
else:
    print('Ejecutar optimizacion_milp.py para generar el plan.')

### 6. Conclusiones de la Fase 2
1. **Mejor Modelo:** `HistGradientBoosting` logró el mejor desempeño con **F1-Score de 0.623** y **ROC-AUC de 0.785**, superando ampliamente a los baselines lineal y trivial.
2. **Variables Determinantes:** El índice NBI rural, la dispersión rural y la infraestructura instalada (sedes ESE) son los principales predictores de la brecha asistencial.
3. **Integración Prescriptiva:** La optimización MILP asigna eficazmente los recursos móviles minimizando distancias de viaje y priorizando comunidades indígenas y rurales.